In [1]:
# Set up all required functions
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.functions import dense_rank, desc,when
from pyspark.sql.functions import dense_rank, desc
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,CharType,ArrayType
from itertools import chain

In [2]:
# Fresh kernel — Cell 1
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.8-hotspot"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ["PATH"]
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = os.path.join(os.environ["HADOOP_HOME"], "bin") + os.pathsep + os.environ["PATH"]

In [3]:
spark = SparkSession.builder \
    .appName("pyspark_ylep") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.local.dir", "D:/spark_tmp") \
    .getOrCreate()

print("Spark version:", spark.version)  # confirm JVM is alive

c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version: 4.2.0


In [25]:
df_read_review_data = spark.read.json("D:\\A_Data_engineer\\Project_Yelp\\Yelp JSON\\yelp_academic_dataset_review.json")

In [26]:
df_dim_user = spark.read.parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/dim_user")

In [27]:
df_dim_business = spark.read.parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/dim_business")

In [28]:
df_dim_date = spark.read.parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/dim_date")

In [29]:
df_dim_tips = spark.read.parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/dim_tips")

In [30]:
df_dim_reviwe = spark.read.parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/dim_review")

In [31]:
df_read_review_data.columns

['business_id',
 'cool',
 'date',
 'funny',
 'review_id',
 'stars',
 'text',
 'useful',
 'user_id']

In [32]:
df_clean_review = df_read_review_data.withColumn("clean_useful",F.when( ( F.col("useful").try_cast("int") >= 0 ) ,F.col("useful")).otherwise(0) )\
                                     .withColumn("clean_cool",F.when( ( F.col("cool").try_cast("int") >= 0 ) ,F.col("cool")).otherwise(0) )\
                                     .withColumn("clean_stars",F.when( ( F.col("stars").try_cast("int") >= 0 ) ,F.col("stars")).otherwise(0) )\
                                     .withColumn("clean_funny",F.when( ( F.col("funny").try_cast("int") >= 0 ) ,F.col("funny")).otherwise(0) )\
                                     .drop(F.col("cool"),F.col("funny"),F.col("stars"),F.col("useful"),F.col("text"))


In [40]:
df_clean_review.count()

6990280

In [33]:
df_join_user_review = df_clean_review.join(df_dim_user,on=df_clean_review["user_id"] == df_dim_user["User_id"],how="inner")\
                   .select(df_dim_user["User_pk_sk"],df_clean_review["*"]).drop(df_clean_review["user_id"])

In [34]:
df_join_user_tips_with_business = df_join_user_review.join(df_dim_business , on = df_join_user_review["business_id"] == df_dim_business["Business_id"] ,how="inner")\
                                  .select(df_dim_business["Business_pk_sk"],df_join_user_review["*"]).drop(df_join_user_review["business_id"])

In [35]:
df_join_date_with_dim_date = df_join_user_tips_with_business.join(df_dim_date , on = df_join_user_tips_with_business["date"] == df_dim_date["Date"] ,how="inner")\
                             .select(df_dim_date["Date_pk_sk"],df_join_user_tips_with_business["*"]  ).drop(df_join_user_tips_with_business["date"])

In [36]:
df_get_review_pk  = df_join_date_with_dim_date.join(df_dim_reviwe , on = df_join_date_with_dim_date["review_id"] == df_dim_reviwe["review_id"] ,how="inner")\
                  .select(df_dim_reviwe["review_pk_sk"],df_join_date_with_dim_date["*"]  ).drop(df_join_date_with_dim_date["review_id"])

In [42]:
df_get_review_pk.write.mode("overwrite").parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/fact_review")

In [37]:
df_get_review_pk.show()

+------------+-----------+--------------+------------+------------+----------+-----------+-----------+
|review_pk_sk| Date_pk_sk|Business_pk_sk|  User_pk_sk|clean_useful|clean_cool|clean_stars|clean_funny|
+------------+-----------+--------------+------------+------------+----------+-----------+-----------+
|171798696217|42949836947|   51539613882|  8589944074|           0|         0|        5.0|          0|
|223338440388|34359998089|   60129552685|       23392|           4|         4|        5.0|          0|
| 77309469418|34359976044|   25769805259|        4593|           6|         3|        4.0|          1|
|266288107125| 8590157014|   68719488992|188978677941|           1|         0|        5.0|          0|
|180388678724|17180345034|   51539615149| 17179880057|           2|         1|        5.0|          0|
|266288084829|17180210247|   68719487525|188978656172|           0|         0|        5.0|          0|
|309237697091|25769827718|   94489287825|       11591|           4|      

In [38]:
df_get_review_pk.count()

6874447

In [41]:
6990280 - 6874447

115833

In [ ]:

# list  of employee data with 5 row values
data =[["1", "sravan", "company 1"],
       ["3", "bobby", "company 3"],
       ["2", "ojaswi", "company 2"],
       ["1", "sravan", "company 1"],
       ["3", "bobby", "company 3"],
       ["4", "rohith", "company 2"],
       ["ee", "gnanesh", "company 1"]]

# specify column names
columns = ['Employee ID','Employee NAME','Company Name']

# creating a dataframe from the lists of data
dataframe = spark.createDataFrame(data,columns)



In [9]:
df_clean_useful = dataframe.withColumn("clean_useful",F.when( ( F.col("Employee ID").try_cast("int") >= 0 ) ,F.col("Employee ID")).otherwise(0) )#.drop(F.col("useful"))

In [10]:
df_clean_useful.show()

+-----------+-------------+------------+------------+
|Employee ID|Employee NAME|Company Name|clean_useful|
+-----------+-------------+------------+------------+
|          1|       sravan|   company 1|           1|
|          3|        bobby|   company 3|           3|
|          2|       ojaswi|   company 2|           2|
|          1|       sravan|   company 1|           1|
|          3|        bobby|   company 3|           3|
|          4|       rohith|   company 2|           4|
|         ee|      gnanesh|   company 1|           0|
+-----------+-------------+------------+------------+

